In [1]:
import pandas as pd 

In [2]:
# Inisialisasi Data
df = pd.read_csv('data_raw/messy_ecommerce_sales_data.csv', sep=',')
df.columns = df.columns.str.strip() # Menghapus whitespaces pada kolom/header tabel (jika ada)

In [3]:
# 1. Inspeksi & Profiling Data
print("--- Info Data Awal ---")
print(df.info())

--- Info Data Awal ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              103 non-null    int64  
 1   Customer_Name   103 non-null    object 
 2   Order_ID        103 non-null    object 
 3   Order_Date      103 non-null    object 
 4   Product         103 non-null    object 
 5   Category        95 non-null     object 
 6   Quantity        98 non-null     object 
 7   Price           98 non-null     object 
 8   Payment_Method  103 non-null    object 
 9   Status          103 non-null    object 
 10  Total           89 non-null     float64
dtypes: float64(1), int64(1), object(9)
memory usage: 9.0+ KB
None


In [4]:
# 2. Penanganan Data Hilang (Missing Value)
# Menghapus baris jika 'Order_Date' kosong (karena tanggal transaksi krusial dan sulit ditebak)
df.dropna(subset=['Order_Date'], inplace=True)

# Mengisi kategori yang kosong dengan menyalin kategori dari produk yang sama
df['Category'] = df['Category'].fillna(df.groupby('Product')['Category'].transform('first'))

# Mengisi harga yang kosong dengan menyalin median harga dari produk yang sama
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
df['Price'] = df['Price'].fillna(df.groupby('Product')['Price'].transform('median'))

# Mengisi jumlah barang (Quantity) yang kosong dengan nilai median keseluruhan
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median())

# Memastikan kembali format akhir setelah NaN hilang
df['Quantity'] = df['Quantity'].astype(int)
df['Price'] = df['Price'].round(2)

In [5]:
# 3. Pembersihan Duplikat
df.drop_duplicates(inplace=True)


In [6]:
# 4. Standardisasi Format & Tipe Data
# Mengubah format tanggal (format yang error/salah akan dipaksa menjadi NaT)
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')

# Mengubah angka menjadi numerik (teks seperti 'abd' akan dipaksa menjadi NaN)
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
df['Total'] = pd.to_numeric(df['Total'], errors='coerce')

In [7]:
# 5. Deteksi & Penanganan Outlier
# Mengubah nilai negatif yang tidak sengaja terinput menjadi nilai positif (absolut)
df['Quantity'] = df['Quantity'].abs()
df['Price'] = df['Price'].abs()
df['Total'] = df['Total'].abs()

# Mengganti harga outlier (misal di atas 5000) menjadi NaN agar bisa diimputasi pada tahap selanjutnya
df.loc[df['Price'] > 5000, 'Price'] = float('nan')

In [8]:
# 6. Pembersihan Teks & Rekayasa Fitur Dasar
# Menghapus whitespaces pada header tabel
df.columns = df.columns.str.strip() 

# Menghapus whitespace pada seluruh nilai di kolom bertipe teks
text_cols = df.select_dtypes(include=['object']).columns
df[text_cols] = df[text_cols].apply(lambda x: x.str.strip())

# Standardisasi kapitalisasi huruf
df['Product'] = df['Product'].str.title()
df['Category'] = df['Category'].str.title()

# Memperbaiki typo pada kategori
df['Category'] = df['Category'].replace('Electronic', 'Electronics')

# Menghapus huruf/karakter yang salah input pada kolom angka (misal '2pcs' -> '2')
df['Quantity'] = df['Quantity'].astype(str).str.upper().str.replace(r'[A-Z]', '', regex=True)


In [9]:
# 7. Validasi & Integritas Data
# Validasi Integritas: Menghitung ulang kolom Total untuk memastikan data akurat
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

df['Total'] = df['Quantity'] * df['Price']
df['Total'] = df['Total'].round(2)

# Validasi Aturan Bisnis:
# Jika transaksi Cancelled, tidak ada uang masuk (Total = 0)
df.loc[df['Status'] == 'Cancelled', 'Total'] = 0

# Jika transaksi Returned, uang dikembalikan (Total diubah menjadi minus)
df.loc[df['Status'] == 'Returned', 'Total'] = -df['Total']

df.head(10)

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,2024-11-22,Blender,Home,3,38.00,Cash on Delivery,Shipped,114.00
1,101,Customer_101,ORD-35783,2025-07-05,Smartphone,Electronics,2,547.26,PayPal,Processing,1094.52
2,102,Customer_102,ORD-84355,2024-12-23,Tennis Racket,Sports,1,389.05,PayPal,Delivered,389.05
3,103,Customer_103,ORD-57811,2025-03-19,Science,Books,5,233.92,PayPal,Processing,1169.60
4,104,Customer_104,ORD-93614,2025-10-20,Biography,Books,1,552.51,Cash on Delivery,Processing,552.51
5,105,Customer_105,ORD-22442,2024-11-20,Tennis Racket,Sports,3,122.06,Cash on Delivery,Cancelled,0.00
6,106,Customer_106,ORD-25885,2025-02-02,Blender,Home,3,978.63,Bank Transfer,Processing,2935.89
7,107,Customer_107,ORD-89122,2025-01-03,Biography,Books,5,587.68,PayPal,Returned,-2938.40
8,108,Customer_108,ORD-64400,2025-10-23,Science,Books,1,600.29,Cash on Delivery,Processing,600.29
9,109,Customer_109,ORD-18512,2025-05-03,Tennis Racket,Sports,5,168.34,Credit Card,Shipped,841.70


In [10]:
# Penyimpanan Data
df.to_csv('data_clean/clean_ecommerce_sales_data.csv', index=False)

print("\n--- Info Data Bersih ---")
print(df.info())
print("\nCuplikan Data Akhir:")
print(df.tail())


--- Info Data Bersih ---
<class 'pandas.core.frame.DataFrame'>
Index: 102 entries, 0 to 101
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID              102 non-null    int64         
 1   Customer_Name   102 non-null    object        
 2   Order_ID        102 non-null    object        
 3   Order_Date      100 non-null    datetime64[ns]
 4   Product         102 non-null    object        
 5   Category        102 non-null    object        
 6   Quantity        102 non-null    int64         
 7   Price           101 non-null    float64       
 8   Payment_Method  102 non-null    object        
 9   Status          102 non-null    object        
 10  Total           101 non-null    float64       
dtypes: datetime64[ns](1), float64(2), int64(2), object(6)
memory usage: 13.6+ KB
None

Cuplikan Data Akhir:
      ID Customer_Name   Order_ID Order_Date     Product     Category  \
97   197  Custom